In [2]:
pip install supabase

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires cachetools<6,>=4.0, but you have cachetools 6.2.6 which is incompatible.



  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached h2-4.3.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 2.4 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/531.3 kB ? eta -:--:--
   ---------------------------------------- 531.3/531.3 kB 4.3 MB/s eta 0:00:00
Using cached typing_exten

In [3]:
pip install sentence_transformers

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/612.9 kB ? eta -:--:--
   ---------------------------------------- 612.9/612.9 kB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/10.7 MB 7.5 MB/s eta 0:00:02
   ------ --------------------------------- 1.8/10.7 MB 7.2 MB/s eta 0:00:02
   ---------- ----------------------------- 2.9/10.7 MB 4.4 MB/s eta 0:00:02
   ------------- -------------------------- 3.7/10.7 MB 4.5 MB/s eta 0:00:02
   --------------- ------------------------ 4.2/10.7 MB 4.0 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/10.7 MB 4.1 MB/s eta 0:00:02
   ------------------- -------------------- 5.2/10.7 MB 3.6 MB/s eta 0:00:02
   ------------------- -------------------- 5.2/10.7 MB 3.6 MB/s eta 0:00:02
   ----------

In [ ]:
import os
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_KEY")

supabase: Client = create_client(supabase_url, supabase_key)
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_and_update():
    print("Fetching records with NULL embeddings...")
    
    response = supabase.table("knowledge_base").select("id", "content", "user_id").is_("embedding", None).execute()
    
    data = response.data
    
    if not data:
        print("✅ No records to process. All embeddings are up to date!")
        return

    print(f"Found {len(data)} records to process...")

    for item in data:
        text = item['content']
        record_id = item['id']
        
        embedding = model.encode(text).tolist()
        
        try:
            supabase.table("knowledge_base").update({
                "embedding": embedding
            }).eq("id", record_id).execute()
            print(f"✅ Updated ID {record_id}: '{text[:30]}...'")
        except Exception as e:
            print(f"❌ Error updating ID {record_id}: {e}")

if __name__ == "__main__":
    embed_and_update()

Loading embedding model... (this might take a minute first time)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching records with NULL embeddings...
Found 1 records to process...
✅ Updated ID 5: '20 kg rice in instock...'
